In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [3]:
import cv2 
from ultralytics import YOLO 
from collections import defaultdict

#Load the model ---------------------------> 
model = YOLO('yolo11n.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [4]:
class_list = model.names 
class_list

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [18]:
from collections import defaultdict
import cv2

# 1. Open the video file you want to process
cap = cv2.VideoCapture(
    "/kaggle/input/datasets/harshal63234/video-test/highway_30s.mp4"
)

# 2. Get the video's width, height, and frames-per-second (fps)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 3. Setup a writer to save our new processed video file
out = cv2.VideoWriter(
    "/kaggle/working/output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps if fps > 0 else 30,
    (width, height),
)

# Here we are generating lines
line_y_red = 470
line_y_blue = line_y_red + 100

# Variables to store counting and tracking information
counted_ids_red_to_blue = set()
counted_ids_blue_to_red = set()

# Dictionaries to count objects by class for each direction
count_red_to_blue = defaultdict(int)
count_blue_to_red = defaultdict(int)

# State dictionaries to track which line was crossed first
crossed_red_first = {}
crossed_blue_first = {}

# 4. Loop through the video frame by frame
while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    break  # Stop if the video has ended

  # Run YOLO tracking on the current frame
  results = model.track(frame, persist=True)

  # Generating lines here ------------------>
  cv2.line(frame, (200, line_y_red), (1500, line_y_red), (0, 0, 255), 3)
  cv2.line(frame, (5, line_y_blue), (1800, line_y_blue), (255, 0, 0), 3)

  # Check if any objects were detected and tracked
  if results[0].boxes is not None and results[0].boxes.id is not None:
    # Extract the box coordinates, tracking IDs, and class numbers
    boxes = results[0].boxes.xyxy.cpu()
    track_ids = results[0].boxes.id.int().cpu().tolist()
    class_indices = results[0].boxes.cls.int().cpu().tolist()

    # Go through each detected object one by one
    for box, track_id, class_idx in zip(boxes, track_ids, class_indices):
      x1, y1, x2, y2 = map(int, box)
      cx = (x1 + x2) // 2
      cy = (y1 + y2) // 2

      # Load name of the object
      class_name = class_list[class_idx]

      # Make dot circle in the centre of every object
      cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

      # Draw a green bounding box around the object
      cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

      # Add text showing the object's Track ID and Class name
      label = f"ID: {track_id} Class: {class_name}"
      cv2.putText(
          frame,
          label,
          (x1, y1 - 10),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.6,
          (0, 255, 255),
          2,
      )

      # --- DOWNWARD COUNTING LOGIC (Red -> Blue) ---
      if cy >= line_y_red:
        if track_id not in crossed_red_first:
          crossed_red_first[track_id] = True

      if cy >= line_y_blue:
        if (
            track_id in crossed_red_first
            and track_id not in counted_ids_red_to_blue
        ):
          counted_ids_red_to_blue.add(track_id)
          count_red_to_blue[class_name] += 1

      # --- UPWARD COUNTING LOGIC (Blue -> Red) ---
      if cy <= line_y_blue:
        if track_id not in crossed_blue_first:
          crossed_blue_first[track_id] = True

      if cy <= line_y_red:
        if (
            track_id in crossed_blue_first
            and track_id not in counted_ids_blue_to_red
        ):
          counted_ids_blue_to_red.add(track_id)
          count_blue_to_red[class_name] += 1

  # Display the counts on the frame
  y_offset = 30
  for class_name, count in count_red_to_blue.items():
    cv2.putText(
        frame,
        f"{class_name} (Down): {count}",
        (10, y_offset),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )
    y_offset += 30

  y_offset += 20  # Add spacing for upward counts
  for class_name, count in count_blue_to_red.items():
    cv2.putText(
        frame,
        f"{class_name} (Up): {count}",
        (10, y_offset),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2,
        cv2.LINE_AA,
    )
    y_offset += 30

  # Save the modified frame into our output video file
  out.write(frame)

# 5. Clean up and close everything when finished
cap.release()
out.release()
print("Done! Video saved to /kaggle/working/output.mp4")


0: 384x640 1 car, 12.9ms
Speed: 3.1ms preprocess, 12.9ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 11.8ms
Speed: 2.4ms preprocess, 11.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 11.2ms
Speed: 2.4ms preprocess, 11.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 11.4ms
Speed: 2.4ms preprocess, 11.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 9.5ms
Speed: 2.2ms preprocess, 9.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 truck, 10.2ms
Speed: 2.3ms preprocess, 10.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 truck, 10.1ms
Speed: 2.5ms preprocess, 10.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 truck, 13.2ms
Speed: 2.8ms preprocess, 13.2ms inference, 1.6ms postpr

In [15]:
# import os

# # Path to the file you want to delete
# file_path = "/kaggle/working/output.mp4"

# if os.path.exists(file_path):
#   os.remove(file_path)
#   print("output.mp4 has been successfully deleted!")
# else:
#   print("File not found or already deleted.")

output.mp4 has been successfully deleted!
